# Projet de Machine Learning : Prédiction de Types de Cancer
**Analyse de données cliniques et modélisation prédictive**

L'objectif de ce projet est de développer un modèle de Machine Learning capable de prédire avec précision le type de cancer d'un patient à partir de différents marqueurs biologiques.

Dans un contexte médical, l'erreur de diagnostic a des conséquences critiques. Notre démarche se veut donc rigoureuse, en accordant une importance particulière à la gestion du déséquilibre des données (certains cancers étant beaucoup plus rares que d'autres) et à l'interprétabilité clinique des résultats.

In [1]:
# On charge les librairies

import pandas as pd
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [2]:
# 1. Chargement des données
# Lecture des deux onglets du fichier Excel fourni
train = pd.read_excel('donnee_etu.xlsx', sheet_name='train')
validation = pd.read_excel('donnee_etu.xlsx', sheet_name='Validation')
# Séparation des features (X) et de la cible (y) pour l'entraînement
X_train = train.drop(columns=['Type'])
y_train = train['Type']
# Copie du jeu de validation (la colonne 'Type' est vide et sera prédite)
X_val = validation.drop(columns=['Type']).copy()

## 1. Préparation des données (Preprocessing)

Avant d'entraîner nos modèles, il est indispensable de préparer les données cliniques :
* **Variables quantitatives (Marqueurs 6 à 9 et Âge) :** Les valeurs manquantes sont imputées par la médiane (plus robuste aux valeurs extrêmes cliniques que la moyenne) et standardisées pour que les variables à grande échelle n'écrasent pas les autres.
* **Variables qualitatives (Marqueurs 2 à 5) :** Les valeurs manquantes sont remplacées par le mode (valeur la plus fréquente). Les statuts `POS` et `NEG` sont encodés en variables binaires (1 et 0) pour être exploitables par les algorithmes.

In [3]:
# 2. Préparation des données (Preprocessing)
# Mapping manuel POS/NEG
mapping = {'POS': 1, 'NEG': 0}
categorical_cols = ['marqueur 2', 'marqueur 3', 'marqueur 4', 'marqueur 5']
for col in categorical_cols:
    X_train[col] = X_train[col].map(mapping)
    X_val[col] = X_val[col].map(mapping)
# Mapping manuel : les algorithmes nécessitent des valeurs numériques# Mapping manuel : les algorithmes nécessitent des valeurs numériques
numeric_features = ['age', 'marqueur 6', 'marqueur 7', 'marqueur 8', 'marqueur 9']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])
# Regroupement des transformations
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_cols)
    ])

## 2. Méthode de Fine-Tuning et Protocole d'Optimisation

Pour garantir la fiabilité de nos prédictions sur de nouveaux patients, nous avons mis en place un protocole d'évaluation strict.

### A. Le choix de la métrique : F1-Score Macro
Notre jeu de données est fortement déséquilibré (les types 1, 4 et 6 sont majoritaires, tandis que les types 2 et 3 sont très rares).
Dans ce contexte clinique, optimiser l'**Accuracy** (Exactitude globale) serait un piège dangereux : un modèle qui prédirait toujours la classe majoritaire aurait un bon score mais ignorerait totalement les patients atteints des cancers rares.
Nous avons donc choisi d'optimiser le **F1-Score Macro**. Cette métrique calcule la moyenne des F1-scores de chaque type de cancer indépendamment, forçant ainsi le modèle à être performant sur **toutes les pathologies**, même les plus rares.


### B. Validation Croisée Stratifiée (Stratified K-Fold)
Pour l'évaluation, nous utilisons une **Stratified K-Fold Cross-Validation** (avec 5 plis). Contrairement à une validation croisée classique, la version "stratifiée" garantit que chaque échantillon d'entraînement et de test contient exactement la même proportion de chaque type de cancer que le jeu de données initial. C'est indispensable pour que le modèle puisse s'entraîner correctement sur les classes minoritaires à chaque itération.


### C. Sélection des Hyperparamètres : GridSearchCV
Conformément aux bonnes pratiques d'optimisation, la stratégie de recherche s'appuie sur une **GridSearchCV** couplée à notre validation croisée (5-fold CV) pour éviter tout surapprentissage.

**Justification de la méthode :**
* **Taille de l'espace de recherche :** Nous optimisons un nombre restreint d'hyperparamètres (2 à 3 par modèle, comme `n_estimators` et `max_depth` pour le Random Forest). L'algorithme de décision classique recommande le **Grid Search** pour les petits espaces de recherche, car il garantit une exploration exhaustive et reproductible sans faire exploser la complexité computationnelle.
* Si nous avions dû optimiser un modèle plus complexe (ex: plus de 5 hyperparamètres sur un XGBoost), nous serions passés sur une Optimisation Bayésienne (via Optuna) pour des raisons de coût de calcul, mais le Grid Search est ici l'approche la plus directe et la plus certaine.


In [4]:
# 3. Modèles et grilles d'hyperparamètres
# Définition des 3 modèles avec random_state=42 pour garantir une stricte reproductibilité
models = {
    'Logistic Regression': {
        'model': LogisticRegression(multi_class='multinomial', max_iter=2000, random_state=42),
        'params': {'classifier__C': [0.1, 1, 10], 'classifier__class_weight': [None, 'balanced']}
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {'classifier__n_estimators': [50, 100, 200], 'classifier__max_depth': [None, 10, 20], 'classifier__class_weight': [None, 'balanced']}
    },
    'SVM': {
        'model': SVC(random_state=42),
        'params': {'classifier__C': [0.1, 1, 10], 'classifier__gamma': ['scale', 'auto'], 'classifier__class_weight': [None, 'balanced']}
    }
}

In [5]:
# 4. Protocole de Validation
# Validation croisée stratifiée : maintient la proportion des cancers rares dans chaque pli
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Métrique : F1-Score Macro pour donner le même poids aux classes minoritaires qu'aux majoritaires
scorer = make_scorer(f1_score, average='macro')

best_model_name = ""
best_model_score = 0
best_model_pipeline = None

In [6]:
# 5. Entraînement et Sélection
print("Recherche des meilleurs hyperparamètres en cours...")
for name, config in models.items():
  # On intègre le preprocessor et le modèle dans un même Pipeline global
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', config['model'])])
    # n_jobs=-1 permet d'utiliser tous les coeurs du processeur pour accélérer le calcul
    grid = GridSearchCV(pipeline, config['params'], cv=cv, scoring=scorer, n_jobs=-1)
    grid.fit(X_train, y_train)

    print(f"{name} -> Meilleur F1-Macro : {grid.best_score_:.4f} (Params: {grid.best_params_})")
    # Sauvegarde du meilleur modèle global
    if grid.best_score_ > best_model_score:
        best_model_score = grid.best_score_
        best_model_name = name
        best_model_pipeline = grid.best_estimator_

print(f"\n---> Modèle retenu : {best_model_name} avec un F1-Macro de {best_model_score:.4f}")

Recherche des meilleurs hyperparamètres en cours...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression -> Meilleur F1-Macro : 0.7904 (Params: {'classifier__C': 10, 'classifier__class_weight': None})
Random Forest -> Meilleur F1-Macro : 0.8124 (Params: {'classifier__class_weight': 'balanced', 'classifier__max_depth': None, 'classifier__n_estimators': 100})
SVM -> Meilleur F1-Macro : 0.7814 (Params: {'classifier__C': 10, 'classifier__class_weight': None, 'classifier__gamma': 'auto'})

---> Modèle retenu : Random Forest avec un F1-Macro de 0.8124


## 3. Sélection du Meilleur Modèle et Argumentaire Clinique

Après comparaison approfondie de la **Régression Logistique**, du **SVM** et du **Random Forest**, c'est ce dernier qui s'impose avec le meilleur F1-Score Macro.

**Justification du choix final dans un contexte clinique :**
1. **Performance sur les minorités :** Le Random Forest (couplé à l'hyperparamètre `class_weight='balanced'`) a parfaitement su pénaliser les erreurs faites sur les classes minoritaires, offrant la meilleure détection globale.
2. **Explicabilité médicale :** Contrairement au SVM qui est souvent perçu comme une "boîte noire", le Random Forest permet d'extraire l'importance des variables (Feature Importance). Il est capable d'indiquer aux cliniciens quels sont les marqueurs biologiques qui ont le plus pesé dans la décision algorithmique. Cette transparence est une exigence absolue pour qu'un outil d'aide à la décision soit accepté et validé par le corps médical.

Le modèle final a donc été utilisé pour générer les prédictions finales sur notre fichier de validation.

In [7]:
# 6. Génération du fichier final
predictions = best_model_pipeline.predict(X_val)
validation['Type'] = predictions
validation.to_csv('Validation_rempli_final.csv', index=False)
print("Prédictions terminées. Fichier 'Validation_rempli_final.csv' généré.")

Prédictions terminées. Fichier 'Validation_rempli_final.csv' généré.
